In [1]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio faiss-cpu sentence-transformers
print("Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 40.3 MB/s eta 0:00:00
Done!


In [2]:
!pip install -q transformers torch
print("Done!")

Done!


In [3]:
from google.colab import files
print("Upload chunks.pkl and knowledge_base.index")
uploaded = files.upload()

Upload chunks.pkl and knowledge_base.index


Saving chunks.pkl to chunks.pkl
Saving knowledge_base.index to knowledge_base.index


In [4]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import faiss
import numpy as np
import pickle
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import nest_asyncio
import uvicorn
from pyngrok import ngrok

# Load retriever
print("Loading retriever...")
st_model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('knowledge_base.index')
with open('chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)
print(f"Retriever ready! {index.ntotal} vectors")

# Load DeBERTa
print("Loading DeBERTa...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained("NiviG/hallucination-detector")
model = AutoModelForSequenceClassification.from_pretrained("NiviG/hallucination-detector")
model = model.float().to(device)
model.eval()
print(f"DeBERTa ready on {device}!")

labels = {0: 'FACTUAL', 1: 'UNCERTAIN', 2: 'HALLUCINATION'}

# FastAPI app
app = FastAPI(title="Hallucination Detector API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

class CheckRequest(BaseModel):
    question: str
    ai_answer: str

@app.get("/")
def root():
    return {"message": "Hallucination Detector API", "status": "running"}

@app.get("/health")
def health():
    return {"status": "healthy"}

@app.post("/check")
def check(request: CheckRequest):
    # Retrieve evidence
    embedding = st_model.encode([request.question])
    distances, indices = index.search(np.array(embedding, dtype=np.float32), 3)
    evidence = [{'text': chunks[i]['text'], 'topic': chunks[i]['topic'], 'score': float(distances[0][j])} for j, i in enumerate(indices[0])]

    # Run DeBERTa
    inputs = tokenizer(
        evidence[0]['text'],
        request.ai_answer,
        truncation=True,
        max_length=256,
        padding='max_length',
        return_tensors='pt'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
        pred_id = int(np.argmax(probs))

    label = labels[pred_id]
    confidence = round(float(probs[pred_id]) * 100, 1)

    if label == 'FACTUAL':
        summary = f"✅ Appears factual ({confidence}% confidence)"
    elif label == 'HALLUCINATION':
        summary = f"⚠️ May be incorrect ({confidence}% confidence)"
    else:
        summary = f"❓ Cannot verify ({confidence}% confidence)"

    return {
        'label': label,
        'confidence': float(probs[pred_id]),
        'scores': {'FACTUAL': float(probs[0]), 'UNCERTAIN': float(probs[1]), 'HALLUCINATION': float(probs[2])},
        'evidence': evidence,
        'summary': summary
    }

print("Starting API...")

Loading retriever...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.34M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DeBERTa ready on cpu!
Starting API...


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn

# Set ngrok token
ngrok.set_auth_token("3GABrZqb5k19mIVqAOoDEsGRFgY_89qcbVohCy167dw2yHJWC")

# Kill any existing tunnels
ngrok.kill()

# Get public URL
ngrok_tunnel = ngrok.connect(8000)
public_url = ngrok_tunnel.public_url
print(f"\n🔥 PUBLIC API URL: {public_url}")
print(f"Test: {public_url}/health")
print(f"Docs: {public_url}/docs\n")

# Run server
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()


🔥 PUBLIC API URL: https://unloving-relation-thumping.ngrok-free.dev
Test: https://unloving-relation-thumping.ngrok-free.dev/health
Docs: https://unloving-relation-thumping.ngrok-free.dev/docs



INFO:     Started server process [1491]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     182.72.168.110:0 - "GET /health HTTP/1.1" 200 OK
INFO:     182.72.168.110:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     182.72.168.110:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     182.72.168.110:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     182.72.168.110:0 - "POST /check HTTP/1.1" 200 OK
